# 🎨 محوّل الصور الفني - اليوم الوطني
# AI Style Transfer - National Day

---

## مرحباً بكم في تجربة التحويل الفني! 🇸🇦

### كيفية الاستخدام | How to Use:

1. **ارفع صورتك** أو التقط صورة بالكاميرا | Upload your photo or take a webcam shot
2. **اختر النمط الفني** المفضل لديك | Choose your preferred artistic style
3. **اضغط على زر التحويل** وانتظر النتيجة | Click Transform and wait for the result
4. **حمّل الصورة** الفنية الجديدة | Download your new artistic image

### الأنماط المتاحة | Available Styles:
- 🌌 **أنمي / كرتون** | Anime / Cartoon
- 🎨 **لوحة مائية** | Watercolor Painting
- 🖼️ **لوحة زيتية** | Oil Painting
- ✏️ **رسم بالرصاص** | Pencil Sketch
- 🇸🇦 **النمط الوطني** | National Day Theme

---
> **ملاحظة:** قد تستغرق عملية التحويل من 10 إلى 30 ثانية حسب النمط المختار
> **Note:** Processing may take 10-30 seconds depending on the chosen style

In [ ]:
# ============================================================
# Cell 2: Install Dependencies
# ============================================================

import subprocess
import sys

print("🔧 Installing dependencies... Please wait...")
print("=" * 60)

# Install core packages
packages = [
    "diffusers==0.21.4",
    "transformers==4.35.0",
    "accelerate==0.24.1",
    "gradio==3.50.2",
    "Pillow==10.1.0",
    "opencv-python-headless==4.8.1.78",
    "torch torchvision --index-url https://download.pytorch.org/whl/cu118",
    "xformers --index-url https://download.pytorch.org/whl/cu118",
    "scipy",
    "safetensors",
    "invisible-watermark",
]

for pkg in packages:
    print(f"📦 Installing {pkg.split('==')[0].split(' ')[0]}...")
    result = subprocess.run(
        f"pip install -q {pkg}",
        shell=True, capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"   ✅ Done")
    else:
        print(f"   ⚠️  Warning: {result.stderr[:100]}")

print("\n" + "=" * 60)
print("✅ All dependencies installed successfully!")

# Check GPU availability
import torch
print("\n🖥️  System Information:")
print(f"   PyTorch version: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print("   ✅ GPU detected - Fast processing enabled!")
else:
    print("   ⚠️  No GPU detected - Processing will be slower on CPU")
    print("   💡 Tip: Go to Runtime > Change runtime type > GPU (T4)")

In [ ]:
# ============================================================
# Cell 3: Import Libraries & Setup
# ============================================================

import torch
import numpy as np
import cv2
import gradio as gr
from PIL import Image, ImageFilter, ImageEnhance, ImageOps
from diffusers import StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler
import warnings
import gc
import os
import time
from io import BytesIO
import base64

warnings.filterwarnings("ignore")

# ── Device setup ──────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("=" * 60)
print("🎨 AI Style Transfer - National Day")
print("   محوّل الصور الفني - اليوم الوطني")
print("=" * 60)
print(f"\n📍 Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"⚡ Using float16 for faster inference")
else:
    print("⚠️  Running on CPU - consider enabling GPU for better performance")
    print("   Runtime > Change runtime type > T4 GPU")

print(f"\n🔢 NumPy: {np.__version__}")
print(f"🖼️  Pillow: {Image.__version__}")
print(f"📹 OpenCV: {cv2.__version__}")

# ── Global model cache ───────────────────────────────────────────────
_sd_pipeline = None

print("\n✅ Libraries loaded successfully!")
print("📌 Models will be loaded on first use (lazy loading)")

In [ ]:
# ============================================================
# Cell 4: Model Loading (Lazy)
# ============================================================

def load_stable_diffusion():
    """Load Stable Diffusion img2img pipeline (lazy, cached)."""
    global _sd_pipeline

    if _sd_pipeline is not None:
        print("✅ SD pipeline already loaded (cached)")
        return _sd_pipeline

    print("⏳ Loading Stable Diffusion v1.5...")
    print("   First load may take 2-5 minutes (downloading ~4 GB)")
    print("   Subsequent runs will be instant (cached)")

    model_id = "runwayml/stable-diffusion-v1-5"

    try:
        _sd_pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
            model_id,
            torch_dtype=DTYPE,
            safety_checker=None,
            requires_safety_checker=False,
            use_auth_token=False,
        )

        # Use fast DPM-Solver scheduler
        _sd_pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
            _sd_pipeline.scheduler.config
        )

        _sd_pipeline = _sd_pipeline.to(DEVICE)

        # Memory optimizations for free T4 tier
        if DEVICE == "cuda":
            _sd_pipeline.enable_attention_slicing()
            _sd_pipeline.enable_vae_slicing()
            try:
                _sd_pipeline.enable_xformers_memory_efficient_attention()
                print("   ✅ xFormers memory optimization enabled")
            except Exception:
                print("   ℹ️  xFormers not available, using standard attention")

        print("✅ Stable Diffusion loaded successfully!")
        return _sd_pipeline

    except Exception as e:
        print(f"❌ Error loading SD model: {e}")
        print("   Falling back to OpenCV-only methods")
        return None


def preload_models():
    """Preload all models with progress display."""
    print("=" * 60)
    print("🚀 Preloading models...")
    print("=" * 60)

    print("\n[1/1] Loading Stable Diffusion img2img pipeline...")
    pipe = load_stable_diffusion()

    if pipe:
        print("\n✅ All models ready!")
        if DEVICE == "cuda":
            used = torch.cuda.memory_allocated() / 1024**3
            total = torch.cuda.get_device_properties(0).total_memory / 1024**3
            print(f"💾 VRAM usage: {used:.1f} / {total:.1f} GB")
    else:
        print("\n⚠️  SD model unavailable. OpenCV styles still work.")

    print("\n📌 Quick-styles (sketch, watercolor, oil) use OpenCV \u2192 no model needed")
    print("📌 Anime style uses Stable Diffusion \u2192 requires first-time download")


# Run preload
preload_models()

In [ ]:
# ============================================================
# Cell 5: Style Transfer Functions
# ============================================================

# ── Helpers ─────────────────────────────────────────────

def pil_to_cv2(pil_img):
    """Convert PIL Image to OpenCV BGR array."""
    return cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)

def cv2_to_pil(cv2_img):
    """Convert OpenCV BGR array to PIL Image."""
    return Image.fromarray(cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))

def resize_for_processing(img, max_size=768):
    """Resize image keeping aspect ratio, max side = max_size."""
    w, h = img.size
    if max(w, h) <= max_size:
        return img
    scale = max_size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    # SD needs multiples of 8
    new_w = (new_w // 8) * 8
    new_h = (new_h // 8) * 8
    return img.resize((new_w, new_h), Image.LANCZOS)

def resize_to_sd(img, target=512):
    """Resize to SD-friendly size (512 or 768)."""
    w, h = img.size
    if w >= h:
        new_w, new_h = target, int(h * target / w)
    else:
        new_w, new_h = int(w * target / h), target
    new_w = max(64, (new_w // 8) * 8)
    new_h = max(64, (new_h // 8) * 8)
    return img.resize((new_w, new_h), Image.LANCZOS)


# ── 1. Anime Style (Stable Diffusion) ───────────────────────

def anime_style(image: Image.Image, strength: float = 0.55) -> Image.Image:
    """Transform image to anime/cartoon style using SD img2img."""
    pipe = load_stable_diffusion()

    if pipe is None:
        print("⚠️  SD unavailable \u2014 using cartoon fallback")
        return cartoon_fallback(image)

    input_img = resize_to_sd(image.convert("RGB"), 512)

    prompt = (
        "anime style, high quality, detailed, masterpiece, "
        "vibrant colors, clean lines, studio ghibli inspired, "
        "digital art, beautiful illustration"
    )
    negative_prompt = (
        "realistic, photographic, blurry, low quality, ugly, "
        "distorted, deformed, noisy, grainy, extra limbs"
    )

    try:
        with torch.autocast(DEVICE if DEVICE == "cuda" else "cpu"):
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=input_img,
                strength=strength,
                guidance_scale=7.5,
                num_inference_steps=25,
            ).images[0]

        if DEVICE == "cuda":
            torch.cuda.empty_cache()
            gc.collect()

        return result

    except torch.cuda.OutOfMemoryError:
        print("⚠️  OOM \u2014 clearing cache and retrying at 512px...")
        torch.cuda.empty_cache()
        gc.collect()
        input_img = resize_to_sd(image.convert("RGB"), 512)
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=input_img,
            strength=strength,
            guidance_scale=7.0,
            num_inference_steps=20,
        ).images[0]
        torch.cuda.empty_cache()
        return result


def cartoon_fallback(image: Image.Image) -> Image.Image:
    """Fast cartoon effect using OpenCV (no SD needed)."""
    img = pil_to_cv2(image)
    img = cv2.resize(img, (512, 512))

    # Bilateral filter for smooth regions
    for _ in range(3):
        img = cv2.bilateralFilter(img, 9, 75, 75)

    # Edge detection
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.adaptiveThreshold(
        cv2.medianBlur(gray, 7), 255,
        cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 2
    )
    edges = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # Color quantization via k-means
    data = np.float32(img).reshape((-1, 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 0.001)
    _, label, center = cv2.kmeans(data, 12, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    center = np.uint8(center)
    quantized = center[label.flatten()].reshape(img.shape)

    cartoon = cv2.bitwise_and(quantized, edges)
    return cv2_to_pil(cartoon)


# ── 2. Watercolor Style (OpenCV) ───────────────────────────

def watercolor_style(image: Image.Image, strength: float = 0.6) -> Image.Image:
    """Watercolor painting effect using OpenCV."""
    img = pil_to_cv2(image)
    h, w = img.shape[:2]
    if max(h, w) > 1024:
        scale = 1024 / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)))

    # Stylize pass
    sigma_s = int(60 + strength * 60)   # 60-120
    sigma_r = 0.3 + strength * 0.25     # 0.3-0.55
    stylized = cv2.stylization(img, sigma_s=sigma_s, sigma_r=sigma_r)

    # Add soft blur for paint bleed
    blurred = cv2.GaussianBlur(stylized, (3, 3), 0)

    # Blend sharpness back slightly
    alpha = 0.7
    result = cv2.addWeighted(stylized, alpha, blurred, 1 - alpha, 0)

    # Boost saturation for vivid watercolor look
    hsv = cv2.cvtColor(result, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * 1.3, 0, 255)
    result = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # Soften edges slightly
    result = cv2.GaussianBlur(result, (3, 3), 0.5)

    pil_result = cv2_to_pil(result)

    # Add paper texture warmth via PIL
    enhancer = ImageEnhance.Color(pil_result)
    pil_result = enhancer.enhance(1.15)

    return pil_result


# ── 3. Oil Painting Style (OpenCV) ─────────────────────────

def oil_painting_style(image: Image.Image, strength: float = 0.6) -> Image.Image:
    """Oil painting effect using xDoG-inspired + color quantization."""
    img = pil_to_cv2(image)
    h, w = img.shape[:2]
    if max(h, w) > 1024:
        scale = 1024 / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)))

    # Detail enhance for oil-like texture
    detail_sigma_s = int(10 + strength * 10)
    result = cv2.detailEnhance(img, sigma_s=detail_sigma_s, sigma_r=0.15)

    # Reduce colors (quantize)
    n_colors = max(6, int(16 - strength * 8))
    data = np.float32(result).reshape((-1, 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 0.5)
    _, label, center = cv2.kmeans(
        data, n_colors, None, criteria, 5, cv2.KMEANS_RANDOM_CENTERS
    )
    center = np.uint8(center)
    quantized = center[label.flatten()].reshape(result.shape)

    # Blend original detail back
    blend = cv2.addWeighted(result, 0.4, quantized, 0.6, 0)

    # Edge-preserving filter for final smoothing
    blend = cv2.edgePreservingFilter(blend, flags=1, sigma_s=30, sigma_r=0.4)

    pil_result = cv2_to_pil(blend)

    # Boost contrast and warmth
    enhancer = ImageEnhance.Contrast(pil_result)
    pil_result = enhancer.enhance(1.1)
    enhancer = ImageEnhance.Color(pil_result)
    pil_result = enhancer.enhance(1.2)

    return pil_result


# ── 4. Pencil Sketch (OpenCV) ──────────────────────────────

def pencil_sketch(image: Image.Image, strength: float = 0.6) -> Image.Image:
    """Pencil sketch effect (grayscale + optional color)."""
    img = pil_to_cv2(image)
    h, w = img.shape[:2]
    if max(h, w) > 1024:
        scale = 1024 / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)))

    shade_factor = 0.03 + strength * 0.05   # 0.03-0.08
    gray_sketch, _ = cv2.pencilSketch(
        img, sigma_s=60, sigma_r=0.07, shade_factor=shade_factor
    )

    # Convert single-channel to 3-channel
    sketch_bgr = cv2.cvtColor(gray_sketch, cv2.COLOR_GRAY2BGR)

    pil_result = cv2_to_pil(sketch_bgr)

    # Slight contrast boost
    enhancer = ImageEnhance.Contrast(pil_result)
    pil_result = enhancer.enhance(1.3)

    return pil_result


# ── 5. National Day Theme (Green overlay) ───────────────────

def national_day_style(image: Image.Image, strength: float = 0.55) -> Image.Image:
    """
    Saudi National Day themed style:
    Watercolor base + green tint overlay + golden accents.
    If SD is available, adds a stylized Arabic pattern.
    """
    # Start with watercolor base
    result = watercolor_style(image, strength=strength * 0.8)

    # Convert to numpy for tinting
    arr = np.array(result).astype(np.float32)

    # Green tint (Saudi flag green: #006C35)
    green_mask = np.zeros_like(arr)
    green_mask[:, :, 1] = 108.0   # G channel
    green_mask[:, :, 0] = 53.0    # B channel (RGB, not BGR here since it's PIL)

    # Soft green overlay (15% opacity)
    tinted = arr * 0.85 + green_mask * 0.15
    tinted = np.clip(tinted, 0, 255).astype(np.uint8)

    result = Image.fromarray(tinted)

    # Add a subtle vignette
    vignette = Image.new("L", result.size, 0)
    import math
    cx, cy = result.size[0] // 2, result.size[1] // 2
    r = math.sqrt(cx**2 + cy**2)
    vig_arr = np.zeros((result.size[1], result.size[0]), dtype=np.float32)
    for y in range(result.size[1]):
        for x in range(result.size[0]):
            d = math.sqrt((x - cx)**2 + (y - cy)**2)
            vig_arr[y, x] = max(0.0, 1.0 - (d / r) * 0.4)
    vig_img = Image.fromarray((vig_arr * 255).astype(np.uint8), "L")

    # Apply vignette
    result_arr = np.array(result).astype(np.float32)
    vig_norm = np.array(vig_img).astype(np.float32) / 255.0
    for c in range(3):
        result_arr[:, :, c] *= vig_norm
    result = Image.fromarray(np.clip(result_arr, 0, 255).astype(np.uint8))

    # Boost saturation
    enhancer = ImageEnhance.Color(result)
    result = enhancer.enhance(1.25)

    return result


# ── 6. Master transform_image function ──────────────────────

STYLE_MAP = {
    "🌌 Anime / \u0623\u0646\u0645\u064a":         ("anime",       anime_style),
    "🎨 Watercolor / \u0645\u0627\u0626\u064a\u0629":   ("watercolor",  watercolor_style),
    "🖼\ufe0f Oil Painting / \u0632\u064a\u062a\u064a\u0629": ("oil",         oil_painting_style),
    "\u270f\ufe0f Pencil Sketch / \u0631\u0635\u0627\u0635": ("sketch",      pencil_sketch),
    "🇸🇦 National Day / \u0648\u0637\u0646\u064a": ("national",    national_day_style),
}

def transform_image(
    image,
    style_name: str,
    strength: float = 0.55,
    progress=gr.Progress(),
) -> tuple:
    """
    Main entry point for the Gradio interface.
    Returns (result_image, status_message).
    """
    if image is None:
        return None, "⚠️ Please upload an image first! | \u0627\u0644\u0631\u062c\u0627\u0621 \u0631\u0641\u0639 \u0635\u0648\u0631\u0629 \u0623\u0648\u0644\u0627\u064b"

    try:
        # Convert to PIL if numpy
        if isinstance(image, np.ndarray):
            pil_image = Image.fromarray(image)
        else:
            pil_image = image

        # Ensure RGB
        pil_image = pil_image.convert("RGB")

        progress(0.1, desc="🖼\ufe0f Preparing image...")
        pil_image = resize_for_processing(pil_image, max_size=768)

        style_key, style_fn = STYLE_MAP.get(style_name, ("anime", anime_style))
        uses_sd = style_key == "anime"

        if uses_sd:
            progress(0.2, desc="🤖 Loading AI model...")
        else:
            progress(0.2, desc="🎨 Applying style filter...")

        start = time.time()
        result = style_fn(pil_image, strength=strength)
        elapsed = time.time() - start

        progress(1.0, desc="✅ Done!")

        status = (
            f"✅ Style applied in {elapsed:.1f}s\n"
            f"   Style: {style_name}\n"
            f"   Size: {result.width}\u00d7{result.height}px"
        )
        return result, status

    except Exception as e:
        err = str(e)
        print(f"Error in transform_image: {err}")
        if "CUDA out of memory" in err or "OutOfMemoryError" in err:
            msg = "❌ GPU out of memory. Try a smaller image or refresh the runtime."
        else:
            msg = f"❌ Error: {err[:200]}"
        return None, msg


print("✅ All style transfer functions defined!")
print("\nAvailable styles:")
for k in STYLE_MAP:
    print(f"  {k}")

In [ ]:
# ============================================================
# Cell 6: Gradio UI — محوّل الصور الفني - اليوم الوطني
# ============================================================

import gradio as gr

# ── Custom CSS ───────────────────────────────────────────
custom_css = """
/* ── Root palette ── */
:root {
    --nd-green:   #006C35;
    --nd-light:   #00A550;
    --nd-gold:    #C8A951;
    --nd-white:   #F8FAF8;
    --nd-dark:    #0A2B1A;
    --radius:     12px;
}

/* ── Page background ── */
body, .gradio-container {
    background: linear-gradient(135deg, #0A2B1A 0%, #0D3B22 50%, #0A2B1A 100%) !important;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;
    min-height: 100vh;
}

/* ── Main header ── */
.main-header {
    background: linear-gradient(135deg, var(--nd-green), var(--nd-dark));
    border: 2px solid var(--nd-gold);
    border-radius: var(--radius);
    padding: 24px 32px;
    text-align: center;
    margin-bottom: 20px;
    box-shadow: 0 8px 32px rgba(0,108,53,0.4);
}
.main-header h1 {
    color: var(--nd-white) !important;
    font-size: 2.4em !important;
    margin: 0 0 8px !important;
    text-shadow: 0 2px 4px rgba(0,0,0,0.5);
    letter-spacing: 1px;
}
.main-header h2 {
    color: var(--nd-gold) !important;
    font-size: 1.5em !important;
    margin: 0 0 16px !important;
}
.main-header p {
    color: #B8D4C0 !important;
    font-size: 1em;
    margin: 0;
}

/* ── Panel cards ── */
.panel-card {
    background: rgba(255,255,255,0.04) !important;
    border: 1px solid rgba(0,165,80,0.25) !important;
    border-radius: var(--radius) !important;
    padding: 16px !important;
    margin-bottom: 16px;
}

/* ── Labels ── */
label, .label-wrap {
    color: #A8D4B8 !important;
    font-weight: 600 !important;
    font-size: 0.95em !important;
}

/* ── Transform button ── */
.transform-btn {
    background: linear-gradient(135deg, var(--nd-green), var(--nd-light)) !important;
    color: white !important;
    font-size: 1.2em !important;
    font-weight: 700 !important;
    border: 2px solid var(--nd-gold) !important;
    border-radius: var(--radius) !important;
    padding: 14px 32px !important;
    width: 100% !important;
    cursor: pointer !important;
    box-shadow: 0 4px 20px rgba(0,108,53,0.5) !important;
    transition: all 0.2s ease !important;
}
.transform-btn:hover {
    box-shadow: 0 6px 28px rgba(0,165,80,0.7) !important;
    transform: translateY(-2px) !important;
}

/* ── Image upload areas ── */
.image-upload {
    border: 2px dashed rgba(0,165,80,0.5) !important;
    border-radius: var(--radius) !important;
    background: rgba(0,108,53,0.06) !important;
    min-height: 280px !important;
}
.image-upload:hover {
    border-color: var(--nd-light) !important;
    background: rgba(0,108,53,0.1) !important;
}

/* ── Status box ── */
.status-box textarea {
    background: rgba(0,0,0,0.3) !important;
    color: #8FD4A8 !important;
    border: 1px solid rgba(0,165,80,0.3) !important;
    border-radius: 8px !important;
    font-family: monospace !important;
    font-size: 0.9em !important;
}

/* ── Slider ── */
input[type=range] {
    accent-color: var(--nd-light) !important;
}

/* ── Radio buttons ── */
.wrap label span {
    color: #C8E6D0 !important;
}

/* ── Instruction box ── */
.instruction-box {
    background: rgba(200, 169, 81, 0.08) !important;
    border: 1px solid rgba(200, 169, 81, 0.3) !important;
    border-radius: var(--radius) !important;
    padding: 14px !important;
    color: #D4C078 !important;
    font-size: 0.92em;
    line-height: 1.7;
    direction: rtl;
}

/* ── Footer ── */
.footer-box {
    text-align: center;
    color: rgba(200,220,210,0.5) !important;
    font-size: 0.85em;
    padding: 12px;
    border-top: 1px solid rgba(0,165,80,0.2);
    margin-top: 12px;
}
"""

# ── Helper: image to download link ──────────────────────────
def image_to_download(img):
    if img is None:
        return None
    buf = BytesIO()
    img.save(buf, format="PNG")
    buf.seek(0)
    return buf

# ── Build Gradio interface ───────────────────────────────
with gr.Blocks(
    title="\u0645\u062d\u0648\u0651\u0644 \u0627\u0644\u0635\u0648\u0631 \u0627\u0644\u0641\u0646\u064a - \u0627\u0644\u064a\u0648\u0645 \u0627\u0644\u0648\u0637\u0646\u064a | AI Style Transfer",
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.green,
        neutral_hue=gr.themes.colors.gray,
    ),
    css=custom_css,
) as demo:

    # ─ Header ─
    gr.HTML("""
    <div class=\"main-header\">
        <h1>\ud83c\udfa8 \u0645\u062d\u0648\u0651\u0644 \u0627\u0644\u0635\u0648\u0631 \u0627\u0644\u0641\u0646\u064a</h1>
        <h2>AI Style Transfer &middot; National Day Edition \ud83c\uddf8\ud83c\udde6</h2>
        <p>\u062d\u0648\u0651\u0644 \u0635\u0648\u0631\u062a\u0643 \u0625\u0644\u0649 \u0644\u0648\u062d\u0629 \u0641\u0646\u064a\u0629 \u0631\u0627\u0626\u0639\u0629 \u0641\u064a \u062b\u0648\u0627\u0646\u0650 &middot; Transform your photo into stunning art in seconds</p>
    </div>
    """)

    with gr.Row():
        # ─ Left column: Controls ─
        with gr.Column(scale=1, min_width=300):
            gr.HTML('<div class="panel-card">')

            gr.Markdown("### \ud83d\udce4 Upload / \u0627\u0644\u062a\u062d\u0645\u064a\u0644", elem_classes=["panel-label"])
            input_image = gr.Image(
                label="Your Photo \u00b7 \u0635\u0648\u0631\u062a\u0643",
                type="pil",
                sources=["upload", "webcam", "clipboard"],
                elem_classes=["image-upload"],
                height=280,
            )

            gr.Markdown("---")
            gr.Markdown("### \ud83c\udfad Choose Style \u00b7 \u0627\u062e\u062a\u0631 \u0627\u0644\u0646\u0645\u0637")

            style_selector = gr.Radio(
                choices=list(STYLE_MAP.keys()),
                value="\ud83c\udf0c Anime / \u0623\u0646\u0645\u064a",
                label="Artistic Style \u00b7 \u0627\u0644\u0646\u0645\u0637 \u0627\u0644\u0641\u0646\u064a",
                interactive=True,
            )

            gr.Markdown("---")
            gr.Markdown("### \u2699\ufe0f Settings \u00b7 \u0627\u0644\u0625\u0639\u062f\u0627\u062f\u0627\u062a")

            strength_slider = gr.Slider(
                minimum=0.30,
                maximum=0.80,
                value=0.55,
                step=0.05,
                label="Style Strength \u00b7 \u0642\u0648\u0629 \u0627\u0644\u062a\u0623\u062b\u064a\u0631",
                info="Low = subtle | High = strong transformation",
                interactive=True,
            )

            transform_btn = gr.Button(
                "\ud83c\udfa8 Transform Now \u00b7 \u062d\u0648\u0651\u0644 \u0627\u0644\u0622\u0646",
                elem_classes=["transform-btn"],
                variant="primary",
            )

            status_box = gr.Textbox(
                label="Status \u00b7 \u0627\u0644\u062d\u0627\u0644\u0629",
                lines=3,
                interactive=False,
                elem_classes=["status-box"],
            )

            gr.HTML('</div>')

            # Arabic instructions
            gr.HTML("""
            <div class=\"instruction-box\">
                <strong>\ud83d\udccb \u062a\u0639\u0644\u064a\u0645\u0627\u062a \u0627\u0644\u0627\u0633\u062a\u062e\u062f\u0627\u0645:</strong><br>
                1\ufe0f\u20e3 \u0627\u0631\u0641\u0639 \u0635\u0648\u0631\u062a\u0643 \u0623\u0648 \u0627\u0633\u062d\u0628\u0647\u0627 \u0625\u0644\u0649 \u0627\u0644\u0645\u0646\u0637\u0642\u0629 \u0623\u0639\u0644\u0627\u0647<br>
                2\ufe0f\u20e3 \u0627\u062e\u062a\u0631 \u0627\u0644\u0646\u0645\u0637 \u0627\u0644\u0641\u0646\u064a \u0627\u0644\u0645\u0641\u0636\u0644<br>
                3\ufe0f\u20e3 \u0627\u0636\u0628\u0637 \u0642\u0648\u0629 \u0627\u0644\u062a\u0623\u062b\u064a\u0631 \u062d\u0633\u0628 \u0631\u063a\u0628\u062a\u0643<br>
                4\ufe0f\u20e3 \u0627\u0636\u063a\u0637 \"\u062d\u0648\u0651\u0644 \u0627\u0644\u0622\u0646\" \u0648\u0627\u0646\u062a\u0638\u0631 \u0627\u0644\u0646\u062a\u064a\u062c\u0629<br>
                5\ufe0f\u20e3 \u0627\u0636\u063a\u0637 \"\u062a\u062d\u0645\u064a\u0644\" \u0644\u062d\u0641\u0638 \u0627\u0644\u0635\u0648\u0631\u0629 \u0627\u0644\u0641\u0646\u064a\u0629<br><br>
                <strong>\ud83d\udca1 \u0646\u0635\u064a\u062d\u0629:</strong> \u0646\u0645\u0637 \u0627\u0644\u0623\u0646\u0645\u064a \u064a\u0633\u062a\u063a\u0631\u0642 20-30 \u062b\u0627\u0646\u064a\u0629\u060c \u0628\u064a\u0646\u0645\u0627 \u0628\u0627\u0642\u064a \u0627\u0644\u0623\u0646\u0645\u0627\u0637 \u062a\u0633\u062a\u063a\u0631\u0642 3-8 \u062b\u0648\u0627\u0646\u064d \u0641\u0642\u0637
            </div>
            """)

        # ─ Right column: Results ─
        with gr.Column(scale=1, min_width=300):
            gr.HTML('<div class="panel-card">')
            gr.Markdown("### \ud83d\uddbc\ufe0f Results \u00b7 \u0627\u0644\u0646\u062a\u064a\u062c\u0629")

            with gr.Tab("\u2728 Result \u00b7 \u0627\u0644\u0646\u062a\u064a\u062c\u0629 \u0627\u0644\u0641\u0646\u064a\u0629"):
                output_image = gr.Image(
                    label="Artistic Result \u00b7 \u0627\u0644\u0635\u0648\u0631\u0629 \u0627\u0644\u0641\u0646\u064a\u0629",
                    type="pil",
                    height=350,
                    interactive=False,
                )
                download_btn = gr.DownloadButton(
                    label="\u2b07\ufe0f Download \u00b7 \u062a\u062d\u0645\u064a\u0644 \u0627\u0644\u0635\u0648\u0631\u0629",
                    variant="secondary",
                    visible=False,
                )

            with gr.Tab("\ud83d\udc41\ufe0f Before / After \u00b7 \u0642\u0628\u0644 \u0648\u0628\u0639\u062f"):
                with gr.Row():
                    before_img = gr.Image(
                        label="Original \u00b7 \u0627\u0644\u0623\u0635\u0644\u064a\u0629",
                        type="pil",
                        interactive=False,
                        height=300,
                    )
                    after_img = gr.Image(
                        label="Styled \u00b7 \u0627\u0644\u0641\u0646\u064a\u0629",
                        type="pil",
                        interactive=False,
                        height=300,
                    )

            gr.HTML('</div>')

            # Style guide
            gr.HTML("""
            <div class=\"panel-card\" style=\"margin-top:8px;\">
                <strong style=\"color:#C8E6D0;\">\ud83d\uddfa\ufe0f Style Guide \u00b7 \u062f\u0644\u064a\u0644 \u0627\u0644\u0623\u0646\u0645\u0627\u0637</strong><br><br>
                <table style=\"width:100%; color:#A8C4B4; font-size:0.88em; border-collapse:collapse;\">
                    <tr style=\"border-bottom:1px solid rgba(0,165,80,0.2);\">
                        <td style=\"padding:6px;\">\ud83c\udf0c Anime</td>
                        <td style=\"padding:6px;\">~25s &middot; GPU</td>
                        <td style=\"padding:6px;\">Studio Ghibli look</td>
                    </tr>
                    <tr style=\"border-bottom:1px solid rgba(0,165,80,0.2);\">
                        <td style=\"padding:6px;\">\ud83c\udfa8 Watercolor</td>
                        <td style=\"padding:6px;\">~3s &middot; Fast</td>
                        <td style=\"padding:6px;\">Soft paint effect</td>
                    </tr>
                    <tr style=\"border-bottom:1px solid rgba(0,165,80,0.2);\">
                        <td style=\"padding:6px;\">\ud83d\uddbc\ufe0f Oil Painting</td>
                        <td style=\"padding:6px;\">~5s &middot; Fast</td>
                        <td style=\"padding:6px;\">Rich textures</td>
                    </tr>
                    <tr style=\"border-bottom:1px solid rgba(0,165,80,0.2);\">
                        <td style=\"padding:6px;\">\u270f\ufe0f Pencil Sketch</td>
                        <td style=\"padding:6px;\">~2s &middot; Fast</td>
                        <td style=\"padding:6px;\">Hand-drawn look</td>
                    </tr>
                    <tr>
                        <td style=\"padding:6px;\">\ud83c\uddf8\ud83c\udde6 National Day</td>
                        <td style=\"padding:6px;\">~5s &middot; Fast</td>
                        <td style=\"padding:6px;\">Green watercolor</td>
                    </tr>
                </table>
            </div>
            """)

    # ── Examples section ─────────────────────────────────────
    gr.Markdown("---")
    gr.Markdown("### \ud83d\udca1 Tips & Examples \u00b7 \u0646\u0635\u0627\u0626\u062d \u0648\u0623\u0645\u062b\u0644\u0629")
    gr.Markdown("""
    **Best results with \u00b7 \u0623\u0641\u0636\u0644 \u0646\u062a\u0627\u0626\u062c \u0645\u0639:**
    - Portrait photos (face clearly visible) \u00b7 \u0635\u0648\u0631 \u0627\u0644\u0648\u062c\u0647 \u0627\u0644\u0648\u0627\u0636\u062d\u0629
    - Well-lit images \u00b7 \u0627\u0644\u0635\u0648\u0631 \u0630\u0627\u062a \u0627\u0644\u0625\u0636\u0627\u0621\u0629 \u0627\u0644\u062c\u064a\u062f\u0629
    - Images 512\u00d7512 to 1024\u00d71024 px \u00b7 \u0627\u0644\u0635\u0648\u0631 \u0628\u064a\u0646 512 \u0648 1024 \u0628\u0643\u0633\u0644

    **Style tips \u00b7 \u0646\u0635\u0627\u0626\u062d \u0627\u0644\u0646\u0645\u0637:**
    - **Anime**: Set strength 0.5-0.65 for best results \u00b7 \u0627\u0636\u0628\u0637 \u0627\u0644\u0642\u0648\u0629 \u0628\u064a\u0646 0.5-0.65
    - **Watercolor**: Works great with landscapes \u00b7 \u0645\u062b\u0627\u0644\u064a\u0629 \u0644\u0644\u0645\u0646\u0627\u0638\u0631 \u0627\u0644\u0637\u0628\u064a\u0639\u064a\u0629
    - **Sketch**: Low strength (0.3) for subtle effect \u00b7 \u0642\u0648\u0629 \u0645\u0646\u062e\u0641\u0636\u0629 \u0644\u062a\u0623\u062b\u064a\u0631 \u062e\u0641\u064a\u0641
    - **National Day**: Perfect for group/portrait photos \u00b7 \u0645\u062b\u0627\u0644\u064a\u0629 \u0644\u0635\u0648\u0631 \u0627\u0644\u0645\u062c\u0645\u0648\u0639\u0627\u062a
    """)

    # ─ Footer ─
    gr.HTML("""
    <div class=\"footer-box\">
        \ud83c\uddf8\ud83c\udde6 Made for Saudi National Day \u00b7 \u0635\u064f\u0646\u0639 \u0628\u0645\u0646\u0627\u0633\u0628\u0629 \u0627\u0644\u064a\u0648\u0645 \u0627\u0644\u0648\u0637\u0646\u064a \u0627\u0644\u0633\u0639\u0648\u062f\u064a &nbsp;|&nbsp;
        Powered by Stable Diffusion + OpenCV &nbsp;|&nbsp;
        Running on Google Colab T4 GPU
    </div>
    """)

    # ── Event handlers ───────────────────────────────────────
    def on_transform(image, style, strength, progress=gr.Progress()):
        result, status = transform_image(image, style, strength, progress)
        # Update before/after views
        before = image if image is not None else None
        after = result
        download_visible = result is not None
        return result, status, before, after, gr.update(visible=download_visible)

    transform_btn.click(
        fn=on_transform,
        inputs=[input_image, style_selector, strength_slider],
        outputs=[output_image, status_box, before_img, after_img, download_btn],
        show_progress=True,
    )

    # Download handler
    def prepare_download(result_img):
        if result_img is None:
            return None
        buf = BytesIO()
        result_img.save(buf, format="PNG")
        buf.seek(0)
        # Save temp file for download
        tmp_path = "/tmp/styled_image.png"
        result_img.save(tmp_path, "PNG")
        return tmp_path

    output_image.change(
        fn=prepare_download,
        inputs=[output_image],
        outputs=[download_btn],
    )

# ── Launch ──────────────────────────────────────────────
print("=" * 60)
print("🚀 Launching Gradio interface...")
print("=" * 60)

demo.queue(max_size=5).launch(
    share=True,          # Creates public URL for event visitors
    debug=False,
    show_error=True,
    quiet=False,
)

In [ ]:
# ============================================================
# Cell 7: Wireless Printing | Tabia Lasilkiya
# ============================================================
# Adds a branded National Day frame and opens the browser print
# dialog so the image is sent to any connected WiFi printer.
# ============================================================

import base64, io, datetime
from PIL import Image, ImageDraw, ImageFont
import gradio as gr


# -- A: Branded print card ---------------------------------

def create_print_card(
    image,
    title_ar='اليوم الوطني السعودي',
    title_en='Saudi National Day',
    subtitle='AI Style Transfer . تحويل الصور بالذكاء الاصطناعي',
    card_size=(1200, 900),
):
    """Wrap the styled image in a branded National Day print card."""
    W, H    = card_size
    BORDER  = 30
    HEADER  = 80
    FOOTER  = 55
    PADDING = 12
    GREEN   = (0, 108, 53)
    GOLD    = (200, 169, 81)

    canvas = Image.new('RGB', (W, H), color=GREEN)

    ix0, iy0 = BORDER, BORDER + HEADER
    ix1, iy1 = W - BORDER, H - BORDER - FOOTER
    canvas.paste(Image.new('RGB', (ix1-ix0, iy1-iy0), 'white'), (ix0, iy0))

    inner_w = ix1 - ix0 - 2*PADDING
    inner_h = iy1 - iy0 - 2*PADDING
    thumb = image.copy().convert('RGB')
    thumb.thumbnail((inner_w, inner_h), Image.LANCZOS)
    px = ix0 + PADDING + (inner_w - thumb.width)  // 2
    py = iy0 + PADDING + (inner_h - thumb.height) // 2
    canvas.paste(thumb, (px, py))

    draw = ImageDraw.Draw(canvas)

    draw.rectangle([BORDER//2, BORDER//2, W-BORDER//2, H-BORDER//2],
                   outline=GOLD, width=3)
    draw.rectangle([BORDER//2+6, BORDER//2+6, W-BORDER//2-6, H-BORDER//2-6],
                   outline=GOLD, width=1)

    def font(px):
        for path in [
            '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
            '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
            '/usr/share/fonts/truetype/freefont/FreeSansBold.ttf',
        ]:
            try:
                return ImageFont.truetype(path, px)
            except Exception:
                pass
        return ImageFont.load_default()

    draw.text((W//2, BORDER + HEADER//2),
              f'{title_ar}  |  {title_en}',
              fill=GOLD, font=font(30), anchor='mm')

    draw.text((W//2, H - BORDER - FOOTER//2 - 10),
              subtitle,
              fill=(180, 220, 180), font=font(19), anchor='mm')
    draw.text((W//2, H - BORDER - FOOTER//2 + 14),
              datetime.datetime.now().strftime('%Y/%m/%d'),
              fill=(150, 190, 160), font=font(16), anchor='mm')

    sq = 14
    for cx, cy in [(BORDER+4, BORDER+4), (W-BORDER-4-sq, BORDER+4),
                   (BORDER+4, H-BORDER-4-sq), (W-BORDER-4-sq, H-BORDER-4-sq)]:
        draw.rectangle([cx, cy, cx+sq, cy+sq], fill=GOLD)

    return canvas


# -- B: Auto-print HTML builder ---------------------------

def build_print_html(image):
    """Encode image as base64 and return HTML with auto-print JavaScript."""
    if image is None:
        return "<p style='color:#f88;'>Upload image first</p>"

    buf = io.BytesIO()
    image.save(buf, format='PNG', dpi=(300, 300))
    b64 = base64.b64encode(buf.getvalue()).decode()

    html_page = (
        '<!DOCTYPE html><html><head><title>National Day Print</title>'
        '<style>'
        'body{margin:0;padding:0;background:#fff;display:flex;'
        'justify-content:center;align-items:center;min-height:100vh;}'
        'img{max-width:100%;max-height:97vh;object-fit:contain;}'
        '@media print{img{width:100%;height:auto;page-break-inside:avoid;}}'
        '</style></head><body>'
        f'<img src="data:image/png;base64,{b64}"'
        ' onload="setTimeout(()=>{window.print();setTimeout(()=>window.close(),2500);},400)">'
        '</body></html>'
    )

    # Escape for JavaScript string
    js_page = html_page.replace('`', r'\`').replace('${', r'\${')

    return (
        '<div style="text-align:center;padding:14px;">'
        '<button onclick="openPrint()" style="'
        'background:linear-gradient(135deg,#006C35,#00A550);'
        'color:white;border:2px solid #C8A951;'
        'padding:14px 38px;font-size:1.15em;font-weight:bold;'
        'border-radius:10px;cursor:pointer;">'
        '&#128424; &nbsp; '
        '\u0637\u0628\u0627\u0639\u0629 \u0639\u0644\u0649 \u0627\u0644\u0637\u0627\u0628\u0639\u0629 \u0627\u0644\u0644\u0627\u0633\u0644\u0643\u064a\u0629'
        ' &nbsp;&middot;&nbsp; Print to Wireless Printer'
        '</button>'
        '<p style="color:#aaa;font-size:0.82em;margin:6px 0 0;">'
        '\u0633\u062a\u0641\u062a\u062d \u0646\u0627\u0641\u0630\u0629 \u0627\u0644\u0637\u0627\u0628\u0639\u0629 \u062a\u0644\u0642\u0627\u0626\u064a\u0627\u064b'
        ' &nbsp;&middot;&nbsp; Print dialog opens automatically'
        '</p></div>'
        f'<script>function openPrint(){{var w=window.open("","_blank","width=980,height=740");w.document.write(`{js_page}`);w.document.close();}}\n</script>'
    )


# -- C: Printing UI ---------------------------------------

print_css = 'body, .gradio-container { background: linear-gradient(135deg, #0A2B1A, #0D3B22) !important; }'

with gr.Blocks(css=print_css, title='Print | Tiba3a') as print_demo:

    gr.HTML("""
    <div style='text-align:center;padding:20px 0 8px;'>
      <span style='color:#C8A951;font-size:1.7em;font-weight:bold;'>
        &#128424; \u0637\u0628\u0627\u0639\u0629 \u0627\u0644\u0635\u0648\u0631\u0629 \u0627\u0644\u0641\u0646\u064a\u0629 &nbsp;&middot;&nbsp; Print Your Art
      </span><br>
      <span style='color:#8FD4A8;font-size:0.95em;'>
        \u0627\u0644\u064a\u0648\u0645 \u0627\u0644\u0648\u0637\u0646\u064a \u0627\u0644\u0633\u0639\u0648\u062f\u064a &#127480;&#127462;
      </span>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            print_input = gr.Image(
                label='Image to Print',
                type='pil',
                sources=['upload', 'clipboard'],
                height=310,
            )
            add_frame = gr.Checkbox(
                label='Add National Day Frame (green + gold)',
                value=True,
            )
            print_btn = gr.Button(
                '&#128424; Print Now',
                variant='primary',
            )

        with gr.Column(scale=1):
            preview = gr.Image(
                label='Print Preview',
                type='pil',
                height=310,
                interactive=False,
            )
            print_html = gr.HTML()

    def on_print(img, frame):
        if img is None:
            return None, "<p style='color:#f88;text-align:center;'>Upload image first</p>"
        card = create_print_card(img) if frame else img
        return card, build_print_html(card)

    print_btn.click(
        fn=on_print,
        inputs=[print_input, add_frame],
        outputs=[preview, print_html],
    )

    gr.HTML("""
    <div style='background:rgba(200,169,81,0.08);border:1px solid rgba(200,169,81,0.3);
                border-radius:10px;padding:14px;margin-top:12px;
                color:#C8A951;font-size:0.88em;line-height:1.8;'>
      <strong>Steps / \u062e\u0637\u0648\u0627\u062a:</strong><br>
      1. Upload the styled image (or paste from clipboard)<br>
      2. Enable 'National Day Frame' for green+gold border<br>
      3. Click Print Now -- print dialog opens automatically<br>
      4. Select your wireless printer and print!<br><br>
      <strong>Tip:</strong> Printer must be on same WiFi with driver installed
    </div>
    """)

print('=' * 60)
print('Wireless Printing Interface ready!')
print('=' * 60)
print_demo.launch(share=True, quiet=True)


# 💡 Tips & Troubleshooting | نصائح واستكشاف الأخطاء

---

## ⚡ Performance Tips | نصائح الأداء

### Getting the Best Results | للحصول على أفضل النتائج:

| Style | Best Input | Strength | Time |
|-------|-----------|----------|------|
| 🌌 Anime | Portrait, face photo | 0.5-0.65 | ~25s |
| 🎨 Watercolor | Landscape, colorful | 0.5-0.7 | ~3s |
| 🖼️ Oil Painting | Portrait, any photo | 0.5-0.7 | ~5s |
| ✏️ Pencil Sketch | Any photo | 0.3-0.6 | ~2s |
| 🇸🇦 National Day | Portrait, group | 0.5-0.65 | ~5s |

---

## 🔧 Common Issues | المشكلات الشائعة

### ❌ "GPU out of memory" Error
```
Solution:
1. Runtime > Restart runtime
2. Re-run all cells
3. Use a smaller image (< 512px)
4. Reduce Style Strength slider
```

### ❌ Anime style is slow or fails
```
Possible causes:
• Model still downloading (first run takes 2-5 min)
• GPU memory full → restart runtime
• No GPU selected → Runtime > Change runtime type > T4

Solutions:
• Wait for download to complete
• Use watercolor/sketch while waiting
• Restart and re-run cells
```

### ❌ "No GPU detected" warning
```
Steps to enable GPU:
1. Runtime → Change runtime type
2. Hardware accelerator → T4 GPU
3. Save → Disconnect and Reconnect
4. Re-run all cells from top
```

### ❌ Gradio link expired
```
Free Gradio share links expire after 72 hours.
Re-run Cell 6 to get a new link.
```

---

## 📱 For Event Staff | لموظفي الفعالية

### Setup Checklist:
- [ ] Open notebook in Google Colab
- [ ] Enable T4 GPU (Runtime > Change runtime type)
- [ ] Run all cells in order (Runtime > Run all)
- [ ] Wait for the public Gradio URL to appear
- [ ] Share the URL or QR code with visitors
- [ ] Keep the Colab tab open (do not close!)

### Visitor Instructions (Arabic):
```
مرحباً بك في تجربة التحويل الفني! 🎨
1. ارفع صورتك أو اسحبها
2. اختر النمط الفني المفضل
3. اضغط "حوّل الآن"
4. انتظر 5-30 ثانية
5. حمّل الصورة الفنية الجديدة!
```

### Quick Reset if Something Goes Wrong:
1. Runtime → Restart and Run All
2. Wait ~5 minutes for models to reload
3. Share the new Gradio URL with visitors

---

## 🌟 Advanced Usage | الاستخدام المتقدم

### Customizing Prompts (Anime Style):
To modify the anime style prompt, edit Cell 5, `anime_style()` function:
```python
prompt = "YOUR CUSTOM PROMPT, anime style, ..."
negative_prompt = "realistic, photo, blurry, ..."
```

### Adding New Styles:
1. Add a new function in Cell 5 following the same pattern
2. Add it to the `STYLE_MAP` dictionary
3. Re-run Cell 5 and Cell 6

### Batch Processing:
For multiple images, call `transform_image()` directly:
```python
from PIL import Image
img = Image.open("your_photo.jpg")
result, status = transform_image(img, "🌌 Anime / أنمي", strength=0.6)
result.save("output.png")
```

---

## 📞 Technical Specs | المواصفات التقنية

- **Base Model**: Stable Diffusion v1.5 (runwayml/stable-diffusion-v1-5)
- **CV Styles**: OpenCV 4.8+ (watercolor, oil, sketch, national day)
- **GPU**: Google Colab T4 (16 GB VRAM)
- **Framework**: Gradio 3.50 + Diffusers 0.21
- **Target**: < 30 seconds per image
- **Max image size**: 768×768 px (auto-resized)

---

*🇸🇦 Developed for Saudi National Day Event · تطوير لفعالية اليوم الوطني السعودي*